In [1]:
import pandas as pd
import os
import google_auth_oauthlib.flow
import googleapiclient.discovery
import googleapiclient.errors
import re

def parse_duration(duration_str):
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration_str)
    if not match:
        return 0
    hours = int(match.group(1)) if match.group(1) else 0
    minutes = int(match.group(2)) if match.group(2) else 0
    seconds = int(match.group(3)) if match.group(3) else 0
    return hours * 3600 + minutes * 60 + seconds

def save_liked_videos_to_csv(liked_videos, pasta_base, filename="curtidas.csv"):
    rows = []
    for video in liked_videos:
        snippet = video.get('snippet', {})
        content_details = video.get('contentDetails', {})
        stats = video.get('statistics', {})
        tags_list = snippet.get("tags", [])
        tags_formatted = "|".join(tags_list) if isinstance(tags_list, list) else ""
        duration_raw = content_details.get("duration", "PT0S")
        duration_seconds = parse_duration(duration_raw)
        rows.append({
            "video_id": video.get("id"),
            "canal_id": snippet.get("channelId"),
            "categoria_id": snippet.get("categoryId"),
            "titulo": snippet.get("title"),
            "tags": tags_formatted,
            "duracao_segundos": duration_seconds,
            "idioma_padrao": snippet.get("defaultLanguage", "n/a"),
            "view_count": int(stats.get("viewCount", 0)),
            "like_count": int(stats.get("likeCount", 0)),
            "data_publicacao": snippet.get("publishedAt")
        })
    os.makedirs(pasta_base, exist_ok=True)
    pd.DataFrame(rows).to_csv(os.path.join(pasta_base, filename), index=False, encoding='utf-8-sig')

def save_subscriptions_csv(subscriptions, pasta_base, filename="inscricoes.csv"):
    rows = []
    for item in subscriptions:
        snippet = item.get('snippet', {})
        content_details = item.get('contentDetails', {})
        rows.append({
            "data_inscricao": snippet.get("publishedAt"),
            "canal_titulo": snippet.get("title"),
            "canal_id": snippet.get("resourceId", {}).get("channelId"),
            "total_videos_canal": content_details.get("totalItemCount"),
            "videos_novos_pendentes": content_details.get("newItemCount")
        })
    os.makedirs(pasta_base, exist_ok=True)
    pd.DataFrame(rows).to_csv(os.path.join(pasta_base, filename), index=False, encoding='utf-8-sig')

def get_all_liked_videos(youtube):
    all_videos = []
    next_page_token = None
    while True:
        request = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            myRating="like",
            maxResults=50,
            pageToken=next_page_token
        )
        response = request.execute()
        items = response.get("items", [])
        all_videos.extend(items)
        next_page_token = response.get("nextPageToken")
        if not next_page_token or len(all_videos) >= 200:
            break
    return all_videos

def get_all_subscribed_channels(youtube):
    all_channels = []
    next_page_token = None
    while True:
        request = youtube.subscriptions().list(
            part="snippet,contentDetails",
            mine=True,
            maxResults=50,
            pageToken=next_page_token
        )
        response = request.execute()
        items = response.get("items", [])
        all_channels.extend(items)
        next_page_token = response.get("nextPageToken")
        if not next_page_token or len(all_channels) >= 200:
            break
    return all_channels

nome = input("Insira seu nome: ")
pasta_usuario = os.path.join("..", "database", nome)

client_secrets_file = os.path.join(pasta_usuario, "client_secret.json")
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
scopes = ["https://www.googleapis.com/auth/youtube.readonly"]
client_secrets_file = os.path.join(pasta_usuario, "client_secret.json")

flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(client_secrets_file, scopes)
credentials = flow.run_local_server(port=0)
youtube = googleapiclient.discovery.build("youtube", "v3", credentials=credentials)

liked_videos = get_all_liked_videos(youtube)
subscribed_channels = get_all_subscribed_channels(youtube)

save_liked_videos_to_csv(liked_videos, pasta_usuario)
save_subscriptions_csv(subscribed_channels, pasta_usuario)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=437323307664-to7ptqeeec0hm4p0smi16cnf3kpltjfe.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A58319%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.readonly&state=81CN6DECjTWeCyhLBF2DCDYK7jZ4oK&code_challenge=c-TJiGhwWq9ICLeaOCmDGcn4NHxb6nllQ7aVTtmbeyE&code_challenge_method=S256&access_type=offline


In [12]:
!pip install google-api-python-client google-auth-oauthlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached google_api_python_client-2.190.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached httplib2-0.31.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached google_auth_httplib2-0.3.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
Using cached google_api_python_client-2.190.0-py3-none-any.whl (14.7 MB)
Using cached google_auth_httplib2-0.3.0-py3-none-any.whl (9.5 kB)
Using cached httplib2-0.31.2-py3-none-any.whl (91 kB)
Using cached uritemplate-4.2.0-py3-none-any.whl (11 kB)

   ---------- ----------------------------- 1/4 [httplib2]
   ------------------------------ --------- 3/4 [google-api-python-client]
   ------------------------------ --------- 3/4 [google-api-python-client]
   ------------------------------ --------- 3/4 [google-api-python-client]
   ------------------------------ --------- 3/4 [google-api-python-client]
   ----------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
